In [ ]:
# CELL 1 — Fetch/update source from GitHub
# Change REPOSITORY_URL once after you publish/fork the repository.

from pathlib import Path
import os
import subprocess

REPOSITORY_URL = os.environ.get(
    "KAGGLE_DEV_REPOSITORY_URL",
    "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY.git",
)
REPOSITORY_REF = os.environ.get("KAGGLE_DEV_REPOSITORY_REF", "main")
PROJECT_DIR = Path(os.environ.get("KAGGLE_DEV_PROJECT_DIR", "/kaggle/working/kaggle-dev-environment"))

if "YOUR_GITHUB_USERNAME" in REPOSITORY_URL or "YOUR_REPOSITORY" in REPOSITORY_URL:
    raise ValueError(
        "Edit REPOSITORY_URL in Cell 1 (or set KAGGLE_DEV_REPOSITORY_URL) "
        "to your public GitHub repository before running this notebook."
    )

PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / ".git").is_dir():
    # Deterministic update: source matches the selected remote ref exactly.
    # Untracked local files such as .kaggle-dev.env are intentionally kept.
    subprocess.run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPOSITORY_URL], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} exists and is not an empty Git repository. Rename/remove it first.")
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
        REPOSITORY_URL, str(PROJECT_DIR)
    ], check=True)

os.chdir(PROJECT_DIR)
commit = subprocess.check_output(["git", "rev-parse", "--short=12", "HEAD"], text=True).strip()
print(f"Project: {PROJECT_DIR}")
print(f"Ref:     {REPOSITORY_REF}")
print(f"Commit:  {commit}")

In [ ]:
# CELL 2 — User configuration (edit this cell)
# Multiple versions are supported. Add a version to the list and give it a unique port.
# No secrets belong in this configuration.

from pathlib import Path
import re
import shlex

CONFIG = {
    "install": {
        "sqlite": True,
        "postgres": True,
        "redis": True,
        "elastic": True,       # Elastic is large; set False if you do not need it.
        "qdrant": True,
        "mise_tools": True,
    },
    "postgres": {
        "versions": ["18"],   # Example multi-version: ["16", "17", "18"]
        "default": "18",
        "ports": {"18": 5433},
        "install_pgvector": True,
        "auto_enable_pgvector": True,
        "auto_start": ["18"], # Installed versions not listed here remain stopped.
    },
    "redis": {
        "versions": ["8.10.0"],  # Exact upstream releases; e.g. ["7.4.10", "8.10.0"]
        "default": "8.10.0",
        "ports": {"8.10.0": 6379},
        "auto_start": ["8.10.0"],
        "build_jobs": 2,
    },
    "elastic": {
        "versions": ["9.5.0"],   # Exact versions; e.g. ["9.4.2", "9.5.0"]
        "default": "9.5.0",
        "components": ["elasticsearch", "kibana", "logstash"],
        "ports": {
            "9.5.0": {
                "elasticsearch": 9200,
                "kibana": 5601,
                "logstash_api": 9600,
                "logstash_input": 5044,
            }
        },
        "heap": "512m",
        "auto_start": [],        # Recommended on Kaggle: start only when needed.
    },
    "qdrant": {
        "versions": ["1.18.3"],  # Exact releases; e.g. ["1.18.2", "1.18.3"]
        "default": "1.18.3",
        "ports": {"1.18.3": 6333},
        "grpc_ports": {"1.18.3": 6334},
        "enable_grpc": False,
        "auto_start": ["1.18.3"],
        "profile": "auto",
        "qnp_release": "1.0.0",
        "qnp_source_commit": "464cb5dbc1117a8a8a6472d76a10c5e329021156",
    },
    "mise": {
        "mise": "2026.8.1",
        "node": "26.6.0",
        "ruby": "3.4.9",
        "npm": "12.0.2",
        "yarn": "1.22.22",
        "add_bashrc_hook": True,
    },
}

def env_key(version: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]", "_", version.replace(".", "_").replace("-", "_"))

def require(condition, message):
    if not condition:
        raise ValueError(message)

def normalize_versions(section, exact_semver=False):
    versions = [str(v).strip() for v in section["versions"]]
    require(versions and all(versions), "Version lists cannot be empty.")
    require(len(versions) == len(set(versions)), f"Duplicate versions: {versions}")
    if exact_semver:
        require(all(re.fullmatch(r"\d+\.\d+\.\d+", v) for v in versions), f"Exact X.Y.Z versions required: {versions}")
    require(str(section["default"]) in versions, f"default={section['default']} must be present in versions={versions}")
    for v in section.get("auto_start", []):
        require(str(v) in versions, f"auto_start version {v} is not in versions={versions}")
    return versions

pg_versions = normalize_versions(CONFIG["postgres"])
require(all(re.fullmatch(r"\d+", v) and 10 <= int(v) <= 99 for v in pg_versions),
        f"PostgreSQL versions must be major numbers such as 16, 17, 18: {pg_versions}")
redis_versions = normalize_versions(CONFIG["redis"], exact_semver=True)
elastic_versions = normalize_versions(CONFIG["elastic"], exact_semver=True)
qdrant_versions = normalize_versions(CONFIG["qdrant"], exact_semver=True)
require(re.fullmatch(r"[0-9a-f]{40}", str(CONFIG["qdrant"]["qnp_source_commit"])),
        "QNP source commit must be a full 40-character lowercase Git SHA.")

# Fill missing port entries deterministically while respecting ports already pinned above.
def next_free(start, used):
    port = int(start)
    while port in used:
        port += 1
    used.add(port)
    return port

pg_used = {int(p) for p in CONFIG["postgres"]["ports"].values()}
for v in pg_versions:
    if v not in CONFIG["postgres"]["ports"]:
        CONFIG["postgres"]["ports"][v] = next_free(5432, pg_used)

redis_used = {int(p) for p in CONFIG["redis"]["ports"].values()}
for v in redis_versions:
    if v not in CONFIG["redis"]["ports"]:
        CONFIG["redis"]["ports"][v] = next_free(6379, redis_used)

elastic_starts = {"elasticsearch": 9200, "kibana": 5601, "logstash_api": 9600, "logstash_input": 5044}
elastic_used = {int(p) for ports in CONFIG["elastic"]["ports"].values() for p in ports.values()}
for v in elastic_versions:
    ep = CONFIG["elastic"]["ports"].setdefault(v, {})
    for name, start in elastic_starts.items():
        if name not in ep:
            ep[name] = next_free(start, elastic_used)

# Qdrant REST/gRPC ports are allocated from one shared set so a future gRPC
# enable cannot silently collide with another configured local service.
all_reserved = set(pg_used) | set(redis_used) | set(elastic_used)
all_reserved.update(int(p) for p in CONFIG["qdrant"]["ports"].values())
all_reserved.update(int(p) for p in CONFIG["qdrant"]["grpc_ports"].values())
for v in qdrant_versions:
    if v not in CONFIG["qdrant"]["ports"]:
        CONFIG["qdrant"]["ports"][v] = next_free(6333, all_reserved)
    if v not in CONFIG["qdrant"]["grpc_ports"]:
        CONFIG["qdrant"]["grpc_ports"][v] = next_free(6334, all_reserved)

allowed_components = {"elasticsearch", "kibana", "logstash"}
components = CONFIG["elastic"]["components"]
require(components and set(components) <= allowed_components, f"Elastic components must be a subset of {sorted(allowed_components)}")
require(re.fullmatch(r"\d+[mMgG]", str(CONFIG["elastic"]["heap"])), "Elastic heap must look like 512m or 1g")

# Global port collision check across every configured service/version.
ports = []
def add_port(label, value):
    value = int(value)
    require(1 <= value <= 65535, f"Invalid port for {label}: {value}")
    ports.append((label, value))

for v in pg_versions:
    add_port(f"postgres:{v}", CONFIG["postgres"]["ports"][v])
for v in redis_versions:
    add_port(f"redis:{v}", CONFIG["redis"]["ports"][v])
for v in elastic_versions:
    ep = CONFIG["elastic"]["ports"][v]
    for name in ("elasticsearch", "kibana", "logstash_api", "logstash_input"):
        add_port(f"elastic:{v}:{name}", ep[name])
for v in qdrant_versions:
    add_port(f"qdrant:{v}:rest", CONFIG["qdrant"]["ports"][v])
    # Reserve configured gRPC ports even while gRPC is disabled so enabling it
    # later cannot silently collide with another configured local service.
    add_port(f"qdrant:{v}:grpc", CONFIG["qdrant"]["grpc_ports"][v])

by_port = {}
for label, port in ports:
    by_port.setdefault(port, []).append(label)
collisions = {port: labels for port, labels in by_port.items() if len(labels) > 1}
require(not collisions, f"Port collisions detected: {collisions}")

def b(value):
    return "1" if value else "0"

def q(value):
    return shlex.quote(str(value))

env = {
    "INSTALL_SQLITE": b(CONFIG["install"]["sqlite"]),
    "INSTALL_POSTGRES": b(CONFIG["install"]["postgres"]),
    "INSTALL_REDIS": b(CONFIG["install"]["redis"]),
    "INSTALL_ELASTIC": b(CONFIG["install"]["elastic"]),
    "INSTALL_QDRANT": b(CONFIG["install"]["qdrant"]),
    "INSTALL_MISE_TOOLS": b(CONFIG["install"]["mise_tools"]),
    "POSTGRES_VERSIONS": " ".join(pg_versions),
    "POSTGRES_DEFAULT_VERSION": CONFIG["postgres"]["default"],
    "POSTGRES_INSTALL_PGVECTOR": b(CONFIG["postgres"]["install_pgvector"]),
    "POSTGRES_AUTO_ENABLE_PGVECTOR": b(CONFIG["postgres"]["auto_enable_pgvector"]),
    "POSTGRES_AUTO_START_VERSIONS": " ".join(CONFIG["postgres"]["auto_start"]),
    "REDIS_VERSIONS": " ".join(redis_versions),
    "REDIS_DEFAULT_VERSION": CONFIG["redis"]["default"],
    "REDIS_AUTO_START_VERSIONS": " ".join(CONFIG["redis"]["auto_start"]),
    "REDIS_BUILD_JOBS": CONFIG["redis"]["build_jobs"],
    "ELASTIC_VERSIONS": " ".join(elastic_versions),
    "ELASTIC_DEFAULT_VERSION": CONFIG["elastic"]["default"],
    "ELASTIC_COMPONENTS": " ".join(components),
    "ELASTIC_AUTO_START_VERSIONS": " ".join(CONFIG["elastic"]["auto_start"]),
    "ELASTIC_HEAP_SIZE": CONFIG["elastic"]["heap"],
    "QDRANT_VERSIONS": " ".join(qdrant_versions),
    "QDRANT_DEFAULT_VERSION": CONFIG["qdrant"]["default"],
    "QDRANT_ENABLE_GRPC": b(CONFIG["qdrant"]["enable_grpc"]),
    "QDRANT_AUTO_START_VERSIONS": " ".join(CONFIG["qdrant"]["auto_start"]),
    "QDRANT_PROFILE": CONFIG["qdrant"]["profile"],
    "QNP_RELEASE": CONFIG["qdrant"]["qnp_release"],
    "QNP_SOURCE_COMMIT": CONFIG["qdrant"]["qnp_source_commit"],
    "MISE_VERSION": CONFIG["mise"]["mise"],
    "TOOL_NODE_VERSION": CONFIG["mise"]["node"],
    "TOOL_RUBY_VERSION": CONFIG["mise"]["ruby"],
    "TOOL_NPM_VERSION": CONFIG["mise"]["npm"],
    "TOOL_YARN_VERSION": CONFIG["mise"]["yarn"],
    "MISE_ADD_BASHRC_HOOK": b(CONFIG["mise"]["add_bashrc_hook"]),
}
for v in pg_versions:
    env[f"POSTGRES_PORT_{env_key(v)}"] = CONFIG["postgres"]["ports"][v]
for v in redis_versions:
    env[f"REDIS_PORT_{env_key(v)}"] = CONFIG["redis"]["ports"][v]
for v in elastic_versions:
    ep = CONFIG["elastic"]["ports"][v]
    key = env_key(v)
    env[f"ELASTIC_PORT_{key}_ELASTICSEARCH"] = ep["elasticsearch"]
    env[f"ELASTIC_PORT_{key}_KIBANA"] = ep["kibana"]
    env[f"ELASTIC_PORT_{key}_LOGSTASH_API"] = ep["logstash_api"]
    env[f"ELASTIC_PORT_{key}_LOGSTASH_INPUT"] = ep["logstash_input"]
for v in qdrant_versions:
    key = env_key(v)
    env[f"QDRANT_PORT_{key}"] = CONFIG["qdrant"]["ports"][v]
    env[f"QDRANT_GRPC_PORT_{key}"] = CONFIG["qdrant"]["grpc_ports"][v]

config_path = Path(PROJECT_DIR) / ".kaggle-dev.env"
with config_path.open("w", encoding="utf-8") as f:
    f.write("# Generated by notebooks/kaggle-dev-bootstrap.ipynb — do not commit.\n")
    for key, value in env.items():
        f.write(f"{key}={q(value)}\n")
config_path.chmod(0o600)

print(f"Wrote {config_path}")
print("PostgreSQL:", pg_versions, CONFIG["postgres"]["ports"])
print("Redis:     ", redis_versions, CONFIG["redis"]["ports"])
print("Elastic:   ", elastic_versions, CONFIG["elastic"]["ports"])
print("Elastic auto-start:", CONFIG["elastic"]["auto_start"] or "disabled")
print("Qdrant:    ", qdrant_versions, CONFIG["qdrant"]["ports"], "gRPC", CONFIG["qdrant"]["grpc_ports"])

## What the first two cells did

Cell 1 fetched/refreshed the public GitHub source. Cell 2 wrote the local, Git-ignored `.kaggle-dev.env` used by every installer. The remaining cells install, verify, and optionally start remote SSH access.


In [ ]:
# CELL 3 — Install/restore selected development tools
import subprocess

subprocess.run(["bash", "install/install-all.sh", "install"], cwd=PROJECT_DIR, check=True)

In [ ]:
# CELL 4 — Verify configuration and installed runtimes
import subprocess

subprocess.run(["bin/kdev", "versions"], cwd=PROJECT_DIR, check=True)
subprocess.run(["bin/kdev", "doctor"], cwd=PROJECT_DIR, check=True)

## Service controls

Examples from a Kaggle terminal / SSH session:

```bash
cd /kaggle/working/kaggle-dev-environment

bin/kdev postgres 18 status
bin/kdev postgres 18 psql

bin/kdev redis 8.10.0 status
bin/kdev redis 8.10.0 cli PING

bin/kdev elastic 9.5.0 start elasticsearch
bin/kdev elastic 9.5.0 status elasticsearch
bin/kdev elastic 9.5.0 stop all
```

Installing many versions is supported, but running every database/Elastic instance at once is usually wasteful on Kaggle.

In [ ]:
# CELL 5 — OPTIONAL: start SSH + ngrok
# Before enabling this, create Kaggle Secrets: SSH_PUBLIC_KEY and NGROK_AUTHTOKEN.
START_SSH = False

if START_SSH:
    subprocess.run(["bash", "setup.sh"], cwd=PROJECT_DIR, check=True)
else:
    print("SSH/ngrok not started. Set START_SSH=True when Kaggle Secrets are configured.")